In [1]:
import pandas as pd

df = pd.read_csv("dataset_cleaned.csv")

# Inspection du lien entre les rayons et leur position
print("=== APERÇU DE LA COLONNE POSITION_RAYON ===")
display(df[['rayon', 'position_rayon']].drop_duplicates().sort_values(by='rayon'))

=== APERÇU DE LA COLONNE POSITION_RAYON ===


,rayon,position_rayon
12,Beurres & Fromages,position_08
682,Beurres & Fromages,position_10
403,Beurres & Fromages,position_04
195,Beurres & Fromages,position_06
13,Beurres & Fromages,position_01
...,...,...
223,Épicerie Sèche,position_26
2216,Épicerie Sèche,position_19
199,Épicerie Sèche,position_03
279,Épicerie Sèche,position_30


In [8]:
import pandas as pd

# Chargement du dataset
df = pd.read_csv("dataset_cleaned.csv")

# Extraction des positions uniques triées
positions_uniques = df['position_rayon'].dropna().unique()
positions_uniques_st = sorted(positions_uniques)

print(f"=== TOTAL : {len(positions_uniques_st)} POSITIONS DISTINCTES DÉTECTÉES ===")
for i, pos in enumerate(positions_uniques_st, 1):
    print(f"{i:02d}. {pos}")

=== TOTAL : 32 POSITIONS DISTINCTES DÉTECTÉES ===
01. position_01
02. position_02
03. position_03
04. position_04
05. position_05
06. position_06
07. position_07
08. position_08
09. position_09
10. position_10
11. position_11
12. position_12
13. position_13
14. position_14
15. position_15
16. position_16
17. position_17
18. position_18
19. position_19
20. position_20
21. position_21
22. position_22
23. position_23
24. position_24
25. position_25
26. position_26
27. position_27
28. position_28
29. position_29
30. position_30
31. position_31
32. position_32


In [10]:
# # Configuration de Pandas pour afficher toutes les lignes
# pd.set_option('display.max_rows', None)

# # Affichage de toutes les combinaisons uniques (Rayon x Position)
# rayon_pos_complet = (df[['rayon', 'position_rayon']]
#                      .drop_duplicates()
#                      .sort_values(by=['rayon', 'position_rayon'])
#                      .reset_index(drop=True))

# display(rayon_pos_complet)

In [11]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Chargement des données
df = pd.read_csv("dataset_cleaned.csv")

print("=== 1. MODÉLISATION DU PLAN DE MASSE (MAPPING DES ALLÉES) ===")

# Attribution de chaque rayon à son allée physique réelle (Exemple de configuration magasin)
plan_allees = {
    'Beurres & Fromages': 1, 'Crèmerie': 1,
    'Boucherie': 2, 'Poissonnerie': 2,
    'Fruits': 3, 'Légumes': 3,
    'Boulangerie': 4, 'Pâtisserie': 4,
    'Céréales & Confitures': 5, 'Café & Thé': 5,
    'Épicerie Sèche': 6, 
    'Huiles & Condiments': 7,
    'Sauces & Soupes': 8, 'Conserves': 8,
    'Snacks': 9, 'Confiserie': 9,
    'Sodas & Boissons': 10, 'Eaux & Jus': 10,
    'Surgelés': 11, 'Glaces': 11,
    'Hygiène & Soins': 12, 'Entretien Ménager': 12, 'Bébé': 12
}

# Conversion en DataFrame
df_allees = pd.DataFrame(list(plan_allees.items()), columns=['rayon', 'allee_physique'])
display(df_allees)

# 2. Construction de la Matrice des Distances Physiques Réelles (en nombre d'allées)
rayons = list(plan_allees.keys())
dist_matrix = pd.DataFrame(index=rayons, columns=rayons, dtype=float)

for r1 in rayons:
    for r2 in rayons:
        dist_matrix.loc[r1, r2] = abs(plan_allees[r1] - plan_allees[r2])

print("\n Matrice des Distances Physiques Réelles créée avec succès !")

=== 1. MODÉLISATION DU PLAN DE MASSE (MAPPING DES ALLÉES) ===


,rayon,allee_physique
0,Beurres & Fromages,1
1,Crèmerie,1
2,Boucherie,2
3,Poissonnerie,2
4,Fruits,3
5,Légumes,3
6,Boulangerie,4
7,Pâtisserie,4
8,Céréales & Confitures,5
9,Café & Thé,5



 Matrice des Distances Physiques Réelles créée avec succès !


In [12]:
print("=== 2. CALCUL DU SCORE SIS AVEC LES DISTANCES D'ALLÉES RÉELLES ===")

# Chargement des règles issues de FP-Growth (Phase 3)
rules = pd.read_csv("regles_association_rayons.csv")

# Filtrage des règles simples (1 rayon -> 1 rayon)
rules_single = rules[
    (~rules['antecedents'].str.contains(',')) & 
    (~rules['consequents'].str.contains(','))
].copy()

# Calcul du SIS
sis_list = []
for idx, row in rules_single.iterrows():
    r1 = row['antecedents']
    r2 = row['consequents']
    lift = row['lift']
    
    if r1 in plan_allees and r2 in plan_allees:
        distance_allees = dist_matrix.loc[r1, r2]
        sis_score = lift * distance_allees
        
        sis_list.append({
            'Rayon_A': r1,
            'Rayon_B': r2,
            'Lift_Affinite': round(lift, 2),
            'Distance_Physique_Allées': int(distance_allees),
            'Score_SIS': round(sis_score, 2)
        })

df_sis = pd.DataFrame(sis_list).sort_values(by='Score_SIS', ascending=False)

print("=== NOUVEAU CLASSEMENT DES INCOHÉRENCES SPATIALES (SIS RECALCULÉ) ===")
display(df_sis.head(10))

# Sauvegarde
df_sis.to_csv("score_incoherence_spatiale_sis_corrige.csv", index=False)


=== 2. CALCUL DU SCORE SIS AVEC LES DISTANCES D'ALLÉES RÉELLES ===
=== NOUVEAU CLASSEMENT DES INCOHÉRENCES SPATIALES (SIS RECALCULÉ) ===


,Rayon_A,Rayon_B,Lift_Affinite,Distance_Physique_Allées,Score_SIS
11,Entretien Ménager,Conserves,1.00,4,4.02
10,Conserves,Entretien Ménager,1.00,4,4.02
9,Sauces & Soupes,Épicerie Sèche,1.04,2,2.07
8,Épicerie Sèche,Sauces & Soupes,1.04,2,2.07
2,Épicerie Sèche,Huiles & Condiments,1.09,1,1.09
3,Huiles & Condiments,Épicerie Sèche,1.09,1,1.09
5,Conserves,Huiles & Condiments,1.07,1,1.07
4,Huiles & Condiments,Conserves,1.07,1,1.07
7,Sauces & Soupes,Huiles & Condiments,1.06,1,1.06
6,Huiles & Condiments,Sauces & Soupes,1.06,1,1.06
